# Per-Residue Structural Variance (PRSV)

This Colab notebook runs `per-residue-structural-variance.R` from the `cianfrocco-lab/nvangos` repository. It installs the R dependencies, loads the PRSV function, reads aligned tubulin PDB/mmCIF structure files, and writes PRSV outputs.

Use the interactive widget panel to choose the alpha- and beta-tubulin chain IDs separately for the base structure and comparison structure.

## 1. Clone the repository

In [ ]:
!if [ ! -d nvangos/.git ]; then git clone --depth 1 https://github.com/cianfrocco-lab/nvangos.git; else git -C nvangos pull --ff-only; fi
%cd nvangos

## 2. Install R dependencies

In [ ]:
%%bash
Rscript - <<'RSCRIPT'
packages <- c("tidyverse", "bio3d", "ggpubr")
installed <- rownames(installed.packages())
missing <- setdiff(packages, installed)
if (length(missing) > 0) {
  install.packages(missing, repos = "https://cloud.r-project.org")
}
RSCRIPT

## 3. Upload input structure files

Upload one base structure file and one comparison structure file from your computer. PDB and mmCIF (`.cif`) files are supported.

In [ ]:
# Upload file 1: base structure file.
from google.colab import files

uploaded_base = files.upload()
if len(uploaded_base) != 1:
    raise ValueError("Please upload exactly one base structure file in this cell.")

BASE_PDB_FILE = next(iter(uploaded_base.keys()))
BASE_PDB_FILE

In [ ]:
# Upload file 2: comparison structure file.
from google.colab import files

uploaded_comparison = files.upload()
if len(uploaded_comparison) != 1:
    raise ValueError("Please upload exactly one comparison structure file in this cell.")

COMPARISON_PDB_FILE = next(iter(uploaded_comparison.keys()))
COMPARISON_PDB_FILE

## 4. Choose files and chain IDs

In [ ]:
import subprocess
import textwrap
import ipywidgets as widgets
from IPython.display import display

if "BASE_PDB_FILE" not in globals() or "COMPARISON_PDB_FILE" not in globals():
    raise RuntimeError("Run both upload cells before creating the chain-selection widgets.")

def clean_cif_token(token):
    token = token.strip()
    if token in {".", "?"}:
        return ""
    if len(token) >= 2 and token[0] == token[-1] and token[0] in {'\"', "'"}:
        return token[1:-1]
    return token

def add_chain(chains, seen, chain):
    label = chain if chain else "(blank)"
    if label not in seen:
        seen.add(label)
        chains.append(label)

def pdb_chain_ids(path):
    options = []
    seen = set()
    with open(path) as handle:
        for line in handle:
            if line.startswith("ATOM"):
                packed_label = line[20:22].strip() if len(line) >= 22 else ""
                fixed_width_chain = line[21].strip() if len(line) > 21 else ""
                label = packed_label if packed_label else fixed_width_chain
                chain = fixed_width_chain if fixed_width_chain else label
                segid = label if len(label) > 1 else ""
                display_label = label if label else "(blank)"
                key = (display_label, chain, segid)
                if key not in seen:
                    seen.add(key)
                    options.append((display_label, (chain, segid)))
    options = sorted(options, key=lambda item: item[0])
    if not options:
        raise ValueError(f"No protein ATOM chain IDs found in {path}")
    return options

def cif_chain_ids(path):
    chains = []
    seen = set()
    with open(path) as handle:
        lines = handle.readlines()

    i = 0
    while i < len(lines):
        if lines[i].strip() != "loop_":
            i += 1
            continue

        headers = []
        j = i + 1
        while j < len(lines) and lines[j].lstrip().startswith("_atom_site."):
            headers.append(lines[j].strip())
            j += 1

        if not headers:
            i += 1
            continue

        auth_idx = headers.index("_atom_site.auth_asym_id") if "_atom_site.auth_asym_id" in headers else None
        label_idx = headers.index("_atom_site.label_asym_id") if "_atom_site.label_asym_id" in headers else None
        chain_idx = auth_idx if auth_idx is not None else label_idx
        group_idx = headers.index("_atom_site.group_PDB") if "_atom_site.group_PDB" in headers else None

        if chain_idx is None:
            i = j
            continue

        k = j
        while k < len(lines):
            stripped = lines[k].strip()
            if not stripped or stripped.startswith("#"):
                break
            if stripped.startswith("_") or stripped == "loop_" or stripped.startswith("data_"):
                break
            fields = stripped.split()
            if len(fields) > chain_idx:
                group = fields[group_idx] if group_idx is not None and len(fields) > group_idx else "ATOM"
                if group == "ATOM":
                    add_chain(chains, seen, clean_cif_token(fields[chain_idx]))
            k += 1
        i = k

    if not chains:
        raise ValueError(f"No mmCIF atom-site chain IDs found in {path}")
    return chains

def python_chain_options(path):
    lower_path = path.lower()
    if lower_path.endswith((".cif", ".mmcif")):
        labels = cif_chain_ids(path)
        return [(label, (label_to_chain(label), "")) for label in labels]
    else:
        return pdb_chain_ids(path)

def label_to_chain(label):
    return "" if label == "(blank)" else label

def bio3d_chain_options(path):
    inspector = r'''
args <- commandArgs(TRUE)
path <- args[[1]]

read_structure <- function(path) {
  lower_path <- tolower(path)
  if (grepl("\\.(cif|mmcif)$", lower_path)) {
    return(bio3d::read.cif(path))
  }
  pdb <- bio3d::read.pdb(path)
  apply_packed_pdb_chain_labels(pdb, path)
}

structure <- read_structure(path)
atoms <- structure$atom
chain <- if ("chain" %in% names(atoms)) atoms$chain else rep("", nrow(atoms))
segid <- if ("segid" %in% names(atoms)) atoms$segid else rep("", nrow(atoms))
chain[is.na(chain)] <- ""
segid[is.na(segid)] <- ""
label <- ifelse(segid != "", segid, chain)
label[label == ""] <- "(blank)"
rows <- unique(data.frame(label = label, chain = chain, segid = segid, stringsAsFactors = FALSE))
write.table(rows, sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
'''
    with open("inspect_structure_chains.R", "w") as handle:
        handle.write(textwrap.dedent(inspector))

    result = subprocess.run(["Rscript", "inspect_structure_chains.R", path], check=True, text=True, capture_output=True)
    options = []
    seen = set()
    for line in result.stdout.splitlines():
        parts = line.split("\t")
        if len(parts) < 3:
            continue
        label, chain, segid = parts[0], parts[1], parts[2]
        key = (label, chain, segid)
        if key not in seen:
            seen.add(key)
            options.append((label, (chain, segid)))
    if not options:
        raise ValueError(f"Bio3D did not report chain IDs for {path}")
    return options

def chain_ids(path):
    if path.lower().endswith((".pdb", ".ent")):
        return python_chain_options(path)
    try:
        return bio3d_chain_options(path)
    except Exception as error:
        print(f"Bio3D chain inspection failed for {path}; using a lightweight parser instead. Details: {error}")
        return python_chain_options(path)

def selected_chain(value):
    return value[0]

def selected_segid(value):
    return value[1]

base_alpha_chain = widgets.Dropdown(description="Base alpha chain:", style={"description_width": "initial"})
base_beta_chain = widgets.Dropdown(description="Base beta chain:", style={"description_width": "initial"})
comparison_alpha_chain = widgets.Dropdown(description="Comparison alpha chain:", style={"description_width": "initial"})
comparison_beta_chain = widgets.Dropdown(description="Comparison beta chain:", style={"description_width": "initial"})
comparison_name = widgets.Text(value="base-vs-comparison", description="Output prefix:", style={"description_width": "initial"})

def set_default_chain(dropdown, options, preferred):
    dropdown.options = options
    preferred_value = options[0][1]
    for label, value in options:
        if label == preferred or label.startswith(preferred):
            preferred_value = value
            break
    dropdown.value = preferred_value

def update_base_chains(*args):
    options = chain_ids(BASE_PDB_FILE)
    set_default_chain(base_alpha_chain, options, "A")
    set_default_chain(base_beta_chain, options, "B")

def update_comparison_chains(*args):
    options = chain_ids(COMPARISON_PDB_FILE)
    set_default_chain(comparison_alpha_chain, options, "A")
    set_default_chain(comparison_beta_chain, options, "B")

update_base_chains()
update_comparison_chains()

display(widgets.VBox([
    widgets.HTML(f"<b>Base structure:</b> {BASE_PDB_FILE}"),
    widgets.HTML(f"<b>Comparison structure:</b> {COMPARISON_PDB_FILE}"),
    widgets.HBox([base_alpha_chain, base_beta_chain]),
    widgets.HBox([comparison_alpha_chain, comparison_beta_chain]),
    comparison_name,
]))

## 5. Run `per-residue-structural-variance.R`

In [ ]:
import os
import subprocess
import textwrap

BASE_PDB = BASE_PDB_FILE
COMPARISON_PDB = COMPARISON_PDB_FILE
BASE_ALPHA_CHAIN = selected_chain(base_alpha_chain.value)
BASE_ALPHA_SEGID = selected_segid(base_alpha_chain.value)
BASE_BETA_CHAIN = selected_chain(base_beta_chain.value)
BASE_BETA_SEGID = selected_segid(base_beta_chain.value)
COMPARISON_ALPHA_CHAIN = selected_chain(comparison_alpha_chain.value)
COMPARISON_ALPHA_SEGID = selected_segid(comparison_alpha_chain.value)
COMPARISON_BETA_CHAIN = selected_chain(comparison_beta_chain.value)
COMPARISON_BETA_SEGID = selected_segid(comparison_beta_chain.value)
COMPARISON_NAME = comparison_name.value.strip() or "prsv-comparison"

OUTPUT_CSV = f"{COMPARISON_NAME}_prsv.csv"
OUTPUT_PNG = f"{COMPARISON_NAME}_prsv.png"

runner = r'''
message("Running PRSV Colab runner version: packed-pdb-labels-2026-06-04")
source("per-residue-structural-variance.R")

if (!exists("apply_packed_pdb_chain_labels")) {
  apply_packed_pdb_chain_labels <- function(pdb, pdb.file) {
    atom.lines <- readLines(pdb.file, warn = FALSE)
    atom.lines <- atom.lines[grepl("^(ATOM|HETATM)", atom.lines)]
    if (length(atom.lines) != nrow(pdb$atom)) {
      warning("Could not apply packed PDB chain labels because ATOM/HETATM line count does not match Bio3D atom count.")
      return(pdb)
    }
    packed.labels <- trimws(substr(atom.lines, 21, 22))
    blank.labels <- is.na(packed.labels) | packed.labels == ""
    packed.labels[blank.labels] <- pdb$atom$chain[blank.labels]
    if (!"segid" %in% names(pdb$atom)) {
      pdb$atom$segid <- ""
    }
    pdb$atom$segid <- ifelse(nchar(packed.labels) > 1, packed.labels, pdb$atom$segid)
    pdb
  }
}

base_path <- Sys.getenv("BASE_PDB")
comparison_path <- Sys.getenv("COMPARISON_PDB")
comparison_name <- Sys.getenv("COMPARISON_NAME")
output_csv <- Sys.getenv("OUTPUT_CSV")
output_png <- Sys.getenv("OUTPUT_PNG")

read_structure <- function(path) {
  lower_path <- tolower(path)
  if (grepl("\\.(cif|mmcif)$", lower_path)) {
    return(bio3d::read.cif(path))
  }
  pdb <- bio3d::read.pdb(path)
  apply_packed_pdb_chain_labels(pdb, path)
}

base <- read_structure(base_path)
comparison <- read_structure(comparison_path)

result <- prsv(base, comparison,
               base.alpha.chain = Sys.getenv("BASE_ALPHA_CHAIN"),
               base.beta.chain = Sys.getenv("BASE_BETA_CHAIN"),
               comp.alpha.chain = Sys.getenv("COMPARISON_ALPHA_CHAIN"),
               comp.beta.chain = Sys.getenv("COMPARISON_BETA_CHAIN"),
               base.alpha.segid = Sys.getenv("BASE_ALPHA_SEGID"),
               base.beta.segid = Sys.getenv("BASE_BETA_SEGID"),
               comp.alpha.segid = Sys.getenv("COMPARISON_ALPHA_SEGID"),
               comp.beta.segid = Sys.getenv("COMPARISON_BETA_SEGID")) %>%
  mutate(Comparison = comparison_name)
write.csv(result, output_csv, row.names = FALSE)

plot <- result %>%
  ggplot(aes(x = Res, y = PRSV, color = Tubulin)) +
  geom_line(linewidth = 0.6) +
  facet_wrap(~Tubulin, ncol = 1, scales = "free_x") +
  labs(x = "Residue", y = expression(bold(paste("Per Residue Structural Variance ", (ring(A)^2)))), title = comparison_name) +
  theme_pubr() +
  theme(plot.title = element_text(face = "bold"))

ggsave(output_png, plot, width = 7, height = 5, dpi = 300)
print(head(result))
message("Wrote ", output_csv)
message("Wrote ", output_png)
'''

with open("run_prsv_colab.R", "w") as handle:
    handle.write(textwrap.dedent(runner))

env = os.environ.copy()
env.update({
    "BASE_PDB": BASE_PDB,
    "COMPARISON_PDB": COMPARISON_PDB,
    "BASE_ALPHA_CHAIN": BASE_ALPHA_CHAIN,
    "BASE_ALPHA_SEGID": BASE_ALPHA_SEGID,
    "BASE_BETA_CHAIN": BASE_BETA_CHAIN,
    "BASE_BETA_SEGID": BASE_BETA_SEGID,
    "COMPARISON_ALPHA_CHAIN": COMPARISON_ALPHA_CHAIN,
    "COMPARISON_ALPHA_SEGID": COMPARISON_ALPHA_SEGID,
    "COMPARISON_BETA_CHAIN": COMPARISON_BETA_CHAIN,
    "COMPARISON_BETA_SEGID": COMPARISON_BETA_SEGID,
    "COMPARISON_NAME": COMPARISON_NAME,
    "OUTPUT_CSV": OUTPUT_CSV,
    "OUTPUT_PNG": OUTPUT_PNG,
})

result = subprocess.run(["Rscript", "run_prsv_colab.R"], check=False, capture_output=True, text=True, env=env)

print("--- R stdout ---")
print(result.stdout if result.stdout else "(empty)")
print("\n--- R stderr ---")
print(result.stderr if result.stderr else "(empty)")

if result.returncode != 0:
    print("\n--- Generated R script ---")
    with open("run_prsv_colab.R") as handle:
        print(handle.read())
    raise RuntimeError(f"R script failed with exit code {result.returncode}. See R stdout/stderr above.")

## 6. Preview and download outputs

In [ ]:
import pandas as pd
from IPython.display import Image, display
from google.colab import files

display(pd.read_csv(OUTPUT_CSV).head())
display(Image(filename=OUTPUT_PNG))

files.download(OUTPUT_CSV)
files.download(OUTPUT_PNG)